In [109]:

from typing import TypedDict, Literal

from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel, Field

load_dotenv()

True

In [110]:
#Feedback state
class FeedbackState(TypedDict):
    user_review: str
    sentiment: Literal["positive", "negative"]

    diagnosis_issue_type: str
    diagnosis_tone: str
    diagnosis_urgency: str

    positive_response: str
    negative_response: str


In [111]:
llm = ChatGroq(model="qwen/qwen3-32b", temperature=0.4)

In [112]:
class DiagnosisOutputParser(BaseModel):
    issue_type: str = Field(
        description="Type of issue mentioned in the user review in string format, either product, service or delivery.")
    tone: str = Field(description="Tone of the user review in string format, either angry, happy or neutral.")
    urgency: str = Field(description="Urgency of the user review in string format, either high, medium or low.")

In [113]:
class SentimentOutputParser(BaseModel):
    sentiment: Literal["positive", "negative"] = Field(
        description="Sentiment of the user review in string format, either positive or negative.")

In [114]:
class ResponseOutputParser(BaseModel):
    response: str = Field(description="Response to the user review in string format.")

In [115]:
def find_sentiment(state: FeedbackState) -> FeedbackState:
    llm_with_parser = llm.with_structured_output(SentimentOutputParser)
    response = llm_with_parser.invoke(
        input=f"Find the sentiment of the following user review and return in structured format - \n User review : {state['user_review']}")
    return {"sentiment": response.sentiment}

In [116]:
def run_diagnosis(state: FeedbackState) -> FeedbackState:
    llm_with_parser = llm.with_structured_output(DiagnosisOutputParser)
    response = llm_with_parser.invoke(
        input=f"Diagnose the following negative user review and return in structured format - \n User review : {state['user_review']}")
    return {
        "diagnosis_issue_type": response.issue_type,
        "diagnosis_tone": response.tone,
        "diagnosis_urgency": response.urgency
    }

In [117]:
def negative_response(state: FeedbackState) -> FeedbackState:
    llm_with_parser = llm.with_structured_output(ResponseOutputParser)
    response = llm_with_parser.invoke(
        input=f"Generate a response to the user review based on the following diagnosis - issue type: {state['diagnosis_issue_type']}, tone: {state['diagnosis_tone']}, urgency: {state['diagnosis_urgency']}. The response should be empathetic and address the user's concerns.")
    return {"negative_response": response.response}

In [118]:
def positive_response(state: FeedbackState) -> FeedbackState:
    llm_with_parser = llm.with_structured_output(ResponseOutputParser)
    response = llm_with_parser.invoke(
        input=f"Generate a response to the user review which is positive. The response should be appreciative and encourage the user to continue engaging with the brand, but dont provide any information which you dont have. - {state['user_review']}")
    return {"positive_response": response.response}

In [119]:
def get_conditional_node(state: FeedbackState) -> Literal["positive_response", "run_diagnosis"]:
    if state["sentiment"] == "positive":
        return "positive_response"
    else:
        return "run_diagnosis"

In [120]:
graph = StateGraph(FeedbackState)

graph.add_node("find_sentiment", find_sentiment)
graph.add_node("run_diagnosis", run_diagnosis)
graph.add_node("negative_response", negative_response)
graph.add_node("positive_response", positive_response)

graph.add_edge(START, "find_sentiment")
graph.add_conditional_edges("find_sentiment", get_conditional_node)
graph.add_edge("run_diagnosis", "negative_response")
graph.add_edge("positive_response", END)

workflow = graph.compile()

In [121]:
inital_state = {
    "user_review": "This fan definitely moves air, and after using it for a while, I can say it has a noticeable effect on the room. Whether that's exactly the effect I was hoping for is harder to say. It does what a fan is expected to do most of the time, though there were moments when I found myself paying more attention to it than I anticipated. Overall, it left an impression, and I haven't felt strongly enough about it to replace it."
}
finalState = workflow.invoke(inital_state)

finalState

{'user_review': "This fan definitely moves air, and after using it for a while, I can say it has a noticeable effect on the room. Whether that's exactly the effect I was hoping for is harder to say. It does what a fan is expected to do most of the time, though there were moments when I found myself paying more attention to it than I anticipated. Overall, it left an impression, and I haven't felt strongly enough about it to replace it.",
 'sentiment': 'positive',
 'positive_response': "Thank you for your thoughtful feedback! We're glad to hear your fan effectively moves air and makes a noticeable impact in your space. It's great to know it meets expectations most of the time, and we appreciate your patience during moments where it caught your attention. Since every room has unique airflow needs, feel free to reach out if you'd like tips on optimizing its performance. We value your loyalty and would love to hear more about your experience in the future. Happy to help!"}